# Assignment 8 -- Robustness Analysis

**Course:** EPA141A Model-Based Decision Making -- Delft University of Technology  
**Model:** JUSTICE  

> **Pre-requisite:** Assignment 7 (Pareto front visualisation). The cross-seed reference set from Assignment 6 is required.

---

## Learning Outcomes

After completing this assignment you will be able to:

1. Explain why policies optimised under a small scenario set may be fragile, and what re-evaluation across a larger ensemble reveals.
2. Run the JUSTICE model programmatically for a given policy and a given FAIR ensemble member, and extract the four objective values.
3. Organise results into a `(n_policies x n_scenarios x n_objectives)` array and compute robustness metrics from it.
4. Compute and interpret **satisficing scores** and **minimax regret** correctly.
5. Identify the minimax-regret policy and articulate its trade-offs relative to the full Pareto front.

## Setup -- Imports, paths, and model constants

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.path as _mpath
import seaborn as sns
from IPython.display import display
from tqdm.notebook import tqdm   # progress bar; pip install tqdm if missing

# Patch matplotlib Path deepcopy
def _patched_path_deepcopy(self, memo=None):
    if memo is None: memo = {}
    new_path = _mpath.Path.__new__(_mpath.Path)
    memo[id(self)] = new_path
    verts = self._vertices.copy()
    codes = self._codes.copy() if self._codes is not None else None
    new_path.__init__(verts, codes,
                      _interpolation_steps=self._interpolation_steps, readonly=False)
    return new_path
_mpath.Path.__deepcopy__ = _patched_path_deepcopy

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib; matplotlib.use("Agg")

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

# Paths
_NOTEBOOK_DIR = os.path.abspath(".")
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))
RESULTS_ROOT  = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))
_CONFIG_DIR   = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../config"))
_PLOTS_DIR    = os.path.join(_NOTEBOOK_DIR, "plots")
os.makedirs(_PLOTS_DIR, exist_ok=True)

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)

# JUSTICE data loaders use relative paths; must run from JUSTICE root.
os.chdir(_JUSTICE_ROOT)

# JUSTICE imports
from justice.model import JUSTICE
from justice.util.data_loader import DataLoader
from justice.util.enumerations import Abatement, DamageFunction, Economy, WelfareFunction
from justice.util.emission_control_constraint import EmissionControlConstraint
from justice.util.model_time import TimeHorizon
from justice.objectives.objective_functions import years_above_temperature_threshold
from solvers.emodps.rbf import RBF

# Config
with open(os.path.join(_CONFIG_DIR, "config_ssp245.json")) as fh:
    _cfg = json.load(fh)

_time_horizon = TimeHorizon(
    start_year=_cfg["start_year"],
    end_year=_cfg["end_year"],
    data_timestep=_cfg["data_timestep"],
    timestep=_cfg["timestep"],
)

N_TIMESTEPS = len(_time_horizon.model_time_horizon)   # 286 (2015-2300, 1-yr steps)
N_REGIONS   = len(DataLoader().REGION_LIST)           # 57 RICE50 regions
N_INPUTS    = _cfg["n_inputs"]                        # 2 (temperature, rate of change)
N_RBFS      = N_INPUTS + 2                            # 4
SCENARIO    = _cfg["reference_ssp_rcp_scenario_index"]  # 2 = SSP2-RCP4.5

EC_START_TS = _time_horizon.year_to_timestep(
    year=_cfg["emission_control_start_year"],
    timestep=_cfg["timestep"],
)

# RBF parameter shapes (needed to slice policy rows correctly)
_rbf_tmp = RBF(n_rbfs=N_RBFS, n_inputs=N_INPUTS, n_outputs=N_REGIONS)
C_SHAPE, R_SHAPE, W_SHAPE = _rbf_tmp.get_shape()

# Scaling constants for RBF inputs
_MAX_TEMP, _MIN_TEMP = 16.0, 0.0
_MAX_DIFF, _MIN_DIFF =  2.0, 0.0

OBJECTIVES = ["welfare", "years_above_2C", "welfare_loss_damage", "welfare_loss_abatement"]
OBJ_LABELS = ["Welfare", "Yrs > 2 C", "WL Damage", "WL Abatement"]

print(f"JUSTICE root : {_JUSTICE_ROOT}")
print(f"Results root : {RESULTS_ROOT}")
print(f"N_TIMESTEPS  : {N_TIMESTEPS}")
print(f"N_REGIONS    : {N_REGIONS}")
print("Setup OK")

---

## Step 1 -- Load the reference set from Assignment 6

The re-evaluation starts from the **cross-seed reference set** built in Assignment 6: `results/reference_set_utilitarian.csv`. This is the epsilon-nondominated set pooled across all seeds.

We extract the **lever columns** (RBF parameters) from the objective columns, since we need to reconstruct the policy for each JUSTICE run.

In [ ]:
# Load reference set
REF_SET_PATH = os.path.join(RESULTS_ROOT, "reference_set_utilitarian.csv")

# Fallback: pre-computed reference set
if not os.path.exists(REF_SET_PATH):
    REF_SET_PATH = os.path.join(
        RESULTS_ROOT, "UTILITARIAN_reference_set.csv"
    )
    print(f"Assignment 6 output not found -- using pre-computed reference set.")

ref_set = pd.read_csv(REF_SET_PATH)
# Drop any penalty solutions (welfare > 1e5 signals a failed evaluation)
ref_set = ref_set[ref_set["welfare"] < 1e5].reset_index(drop=True)

# Identify lever vs objective columns
OPT_OBJECTIVES = ["welfare", "fraction_above_threshold",
                  "welfare_loss_damage", "welfare_loss_abatement"]
LEVER_COLS = [c for c in ref_set.columns if c not in OPT_OBJECTIVES]

print(f"Reference set loaded from: {REF_SET_PATH}")
print(f"  {len(ref_set)} policies  |  {len(LEVER_COLS)} lever columns")
print(f"  Lever columns (first 5): {LEVER_COLS[:5]}")
print()
print("Optimisation objective ranges:")
print(ref_set[OPT_OBJECTIVES].describe().round(3).to_string())

---

## Step 2 -- Re-evaluation using the EMA Workbench

### Why the EMA Workbench?

The EMA Workbench (`perform_experiments`) handles the full factorial experiment -- every policy x every scenario -- with built-in multiprocessing, progress tracking, and a clean result structure. Compared to a manual nested loop, it:

- **Parallelises automatically** across CPU cores via `MultiprocessingEvaluator`.
- **Separates uncertainties from levers**, reflecting the correct conceptual distinction: FAIR ensemble members are *scenarios* (things we cannot control), while RBF parameters are *policies* (things we choose).
- **Returns tidy DataFrames** that are easy to reshape into the `(n_policies x n_scenarios x n_objectives)` array we need for robustness metrics.

### How the model wrapper works

We wrap JUSTICE in a plain Python function whose keyword arguments match the EMA Workbench naming convention:
- One **uncertainty**: `climate_ensemble_index` -- an `IntegerParameter` in `[1, 1000]`. EMA varies this across scenarios.
- **Levers**: the 244 RBF parameters (`center i`, `radii i`, `weights i`). EMA keeps these fixed per policy.

Each call runs one complete JUSTICE simulation (286 timesteps, 1 FAIR member) and returns the four objective values.

> **Runtime note:** 106 policies x 50 scenarios = 5 300 JUSTICE calls. On a modern laptop this takes approximately **15-30 minutes** with multiprocessing. Reduce `N_SCENARIOS` to 10 for a quick test (~3-5 min). Pre-computed results are cached to disk and loaded automatically on subsequent runs.

In [ ]:
from ema_workbench import (
    Model, RealParameter, IntegerParameter, ScalarOutcome,
    Sample, SequentialEvaluator, ema_logging,
)

# Note on evaluators:
# MultiprocessingEvaluator requires the model wrapper to be importable from a
# module file (it uses Python's multiprocessing 'spawn' start method, which
# cannot pickle functions defined interactively in a Jupyter kernel).
# For notebook execution we use SequentialEvaluator.
# If running from a plain Python script (.py), replace with MultiprocessingEvaluator.

ema_logging.log_to_stderr(ema_logging.INFO)

# EMA Workbench model wrapper -- keep this cell unchanged
def model_wrapper_reeval(**kwargs) -> tuple:
    """
    JUSTICE model wrapper for EMA Workbench re-evaluation.

    Called once per (policy, scenario) combination by perform_experiments.
    kwargs contains both lever values (RBF parameters) and the uncertainty
    value (climate_ensemble_index).
    """
    ensemble_index = int(kwargs.pop("climate_ensemble_index"))

    # Reconstruct RBF from lever keyword arguments
    rbf = RBF(n_rbfs=N_RBFS, n_inputs=N_INPUTS, n_outputs=N_REGIONS)
    centers = np.array([kwargs.pop(f"center {i}") for i in range(C_SHAPE[0])])
    radii   = np.array([kwargs.pop(f"radii {i}")  for i in range(R_SHAPE[0])])
    weights = np.array([kwargs.pop(f"weights {i}") for i in range(W_SHAPE[0])])
    rbf.set_decision_vars(np.concatenate([centers, radii, weights]))

    constraint = EmissionControlConstraint(
        max_annual_growth_rate=0.04,
        emission_control_start_timestep=EC_START_TS,
        min_emission_control_rate=0.01,
    )

    model = JUSTICE(
        scenario=SCENARIO,
        climate_ensembles=[ensemble_index],
        economy_type=Economy.NEOCLASSICAL,
        damage_function_type=DamageFunction.KALKUHL,
        abatement_type=Abatement.ENERDATA,
        social_welfare_function_type=WelfareFunction.UTILITARIAN.value[0],
    )
    no_ens = model.no_of_ensembles  # = 1

    ecr             = np.zeros((N_REGIONS, N_TIMESTEPS, no_ens))
    constrained_ecr = np.zeros_like(ecr)
    prev_temp = np.zeros(no_ens)
    diff      = np.zeros(no_ens)

    for t in range(N_TIMESTEPS):
        constrained_ecr[:, t, :] = constraint.constrain_emission_control_rate(
            ecr[:, t, :], t, allow_fallback=False
        )
        model.stepwise_run(
            emission_control_rate=constrained_ecr[:, t, :],
            timestep=t,
            endogenous_savings_rate=True,
        )
        data_t = model.stepwise_evaluate(timestep=t)
        temp = data_t["global_temperature"][t, :]

        if t % 5 == 0:
            diff      = temp - prev_temp
            prev_temp = temp.copy()

        scaled_temp = (temp - _MIN_TEMP) / (_MAX_TEMP - _MIN_TEMP)
        scaled_diff = (diff - _MIN_DIFF) / (_MAX_DIFF - _MIN_DIFF)

        if t < N_TIMESTEPS - 1:
            ecr[:, t + 1, :] = rbf.apply_rbfs(np.array([scaled_temp, scaled_diff]))

    data = model.evaluate()

    welfare = float(np.abs(data["welfare"]))
    welfare = welfare if np.isfinite(welfare) else 1e6

    yrs_above = float(
        years_above_temperature_threshold(data["global_temperature"], threshold=2.0)
    )

    _, _, _, wl_damage = model.welfare_function.calculate_welfare(
        data["damage_cost_per_capita"], welfare_loss=True
    )
    wl_damage = float(np.abs(wl_damage)) if np.isfinite(wl_damage) else 1e6

    _, _, _, wl_abatement = model.welfare_function.calculate_welfare(
        data["abatement_cost_per_capita"], welfare_loss=True
    )
    wl_abatement = float(np.abs(wl_abatement)) if np.isfinite(wl_abatement) else 1e6

    return (welfare, yrs_above, wl_damage, wl_abatement)


# EMA Model definition
# EMA model names must be alphanumeric only (no underscores or hyphens)
ema_model = Model("JUSTICEreeval", function=model_wrapper_reeval)

# One uncertainty: which FAIR ensemble member to use (varied across scenarios)
ema_model.uncertainties = [
    IntegerParameter("climate_ensemble_index", 1, 1000)
]

# Levers: 244 RBF parameters (varied across policies, fixed within each policy)
n_cr = C_SHAPE[0]   # number of center/radii parameters each
n_w  = W_SHAPE[0]   # number of weight parameters
ema_model.levers = (
    [RealParameter(f"center {i}", -1.0, 1.0) for i in range(n_cr)]
    + [RealParameter(f"radii {i}",  0.0, 1.0) for i in range(n_cr)]
    + [RealParameter(f"weights {i}", 0.0, 1.0) for i in range(n_w)]
)

ema_model.outcomes = [
    ScalarOutcome("welfare",                kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("years_above_2C",         kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("welfare_loss_damage",    kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("welfare_loss_abatement", kind=ScalarOutcome.MINIMIZE),
]

print(f"EMA model defined: {len(ema_model.uncertainties)} uncertainty, "
      f"{len(ema_model.levers)} levers, {len(ema_model.outcomes)} outcomes")
print(f"  Levers: {n_cr} centers + {n_cr} radii + {n_w} weights = {n_cr*2 + n_w} total")

### Build Policy and Scenario objects, then run `perform_experiments`

`Policy` objects wrap the RBF lever values for each Pareto solution.  
`Scenario` objects wrap the FAIR ensemble index for each climate trajectory.

`perform_experiments` runs the full factorial: every policy is evaluated in every scenario.  
The result is an `experiments` DataFrame plus an `outcomes` dict.  
We then reshape these into the `(n_policies, n_scenarios, n_objectives)` array used for robustness metrics.

In [ ]:
# Scenario set: 50 well-distributed FAIR ensemble members
# Reduce N_SCENARIOS to 10 for a quick test (~3-5 min).
N_SCENARIOS      = 50
SCENARIO_INDICES = list(np.linspace(1, 1000, N_SCENARIOS, dtype=int))

N_POLICIES   = len(ref_set)
N_OBJECTIVES = len(OBJECTIVES)

# Cache paths
RESULTS_PATH      = os.path.join(
    RESULTS_ROOT,
    f"reeval_utilitarian_{N_POLICIES}p_{N_SCENARIOS}s.npy"
)
EXPERIMENTS_PATH  = os.path.join(
    RESULTS_ROOT,
    f"reeval_utilitarian_{N_POLICIES}p_{N_SCENARIOS}s_experiments.csv"
)

# Build EMA Sample objects for policies (one per Pareto solution)
policies = [
    Sample(f"P{pi}", **{col: float(ref_set.iloc[pi][col]) for col in LEVER_COLS})
    for pi in range(N_POLICIES)
]

# Build EMA Sample objects for scenarios (one per FAIR ensemble member)
scenarios = [
    Sample(f"FAIR_{idx}", climate_ensemble_index=int(idx))
    for idx in SCENARIO_INDICES
]

print(f"Policies     : {N_POLICIES}")
print(f"Scenarios    : {N_SCENARIOS}  (indices: {SCENARIO_INDICES[:3]} ... {SCENARIO_INDICES[-3:]})")
print(f"Results cache: {RESULTS_PATH}")
print(f"Experiments  : {EXPERIMENTS_PATH}")

In [ ]:
if os.path.exists(RESULTS_PATH) and os.path.exists(EXPERIMENTS_PATH):
    # Load pre-computed results
    results     = np.load(RESULTS_PATH)
    experiments = pd.read_csv(EXPERIMENTS_PATH)
    print(f"Loaded pre-computed results: {results.shape}  from {RESULTS_PATH}")

else:
    # Run re-evaluation via EMA Workbench perform_experiments
    # SequentialEvaluator works in Jupyter notebooks.
    with SequentialEvaluator(ema_model) as evaluator:
        experiments, outcomes = evaluator.perform_experiments(
            scenarios=scenarios,
            policies=policies,
        )

    # Reshape outcomes into (n_policies, n_scenarios, n_objectives)
    policy_name_to_idx   = {f"P{pi}": pi for pi in range(N_POLICIES)}
    scenario_name_to_idx = {f"FAIR_{idx}": si
                            for si, idx in enumerate(SCENARIO_INDICES)}

    results = np.full((N_POLICIES, N_SCENARIOS, N_OBJECTIVES), np.nan)

    for row_i, row in experiments.iterrows():
        pi = policy_name_to_idx.get(row["policy"])
        si = scenario_name_to_idx.get(row["scenario"])
        if pi is None or si is None:
            continue
        for oi, obj in enumerate(OBJECTIVES):
            results[pi, si, oi] = outcomes[obj][row_i]

    # Cache to disk
    np.save(RESULTS_PATH, results)
    experiments.to_csv(EXPERIMENTS_PATH, index=False)
    print(f"Re-evaluation complete. Results saved to {RESULTS_PATH}")

print(f"Results array shape : {results.shape}  "
      f"(n_policies={results.shape[0]}, n_scenarios={results.shape[1]}, "
      f"n_objectives={results.shape[2]})")
print(f"NaN entries         : {np.isnan(results).sum()}")

# Quick sanity check: objective ranges
print("\nObjective ranges across all (policy x scenario) evaluations:")
flat = results.reshape(-1, N_OBJECTIVES)
for name, col in zip(OBJECTIVES, flat.T):
    print(f"  {name:<30s}  min={np.nanmin(col):9.2f}  max={np.nanmax(col):9.2f}")

### What to check

- **Shape `(n_policies, n_scenarios, n_objectives)`** -- one row per policy, one column per scenario, 4 objectives.
- **Zero NaN entries** -- if there are failures, check that JUSTICE-main is fully installed and the CWD was changed to the JUSTICE root.
- **Objective ranges across scenarios** -- a large spread on `years_above_2C` means the climate trajectories vary substantially.

---

## Step 3 -- Satisficing analysis

A policy satisfices in a scenario if it meets the threshold on **every** objective. We use the **median across all (policy, scenario) pairs** as the threshold -- a symmetric, data-driven choice.

The **satisficing score** of a policy is the fraction of the 50 scenarios in which it satisfices all objectives.

**Task 3.1** -- Run the cell below, then answer: which objective is the binding constraint (lowest per-objective satisficing rate)?

*(Write your answer here.)*

In [ ]:
# Satisficing: a policy "satisfices" if it meets a threshold in a given scenario.
# Use the median across all (policy x scenario) pairs as the threshold per objective.
#
# Steps:
#   1. flat = results.reshape(-1, N_OBJECTIVES)           # all (policy, scenario) pairs
#   2. thresholds = np.nanmedian(flat, axis=0)            # one threshold per objective
#   3. satisficing = results <= thresholds[np.newaxis, np.newaxis, :]   # broadcast
#   4. sat_score = np.nanmean(satisficing.min(axis=2), axis=1)   # joint: all objectives met

flat       = results.reshape(-1, N_OBJECTIVES)
thresholds = ...   # shape (n_objectives,)
satisficing = ...  # shape (n_policies, n_scenarios, n_objectives)  — bool
sat_score   = ...  # shape (n_policies,)  — fraction of scenarios where ALL objectives met
sorted_idx  = np.argsort(sat_score)[::-1]

print("Satisficing thresholds (median across all policy-scenario pairs):")
for obj, thr in zip(OBJECTIVES, thresholds):
    print(f"  {obj:35s}: {thr:.4g}")

print(f"\nTop 5 policies by joint satisficing score:")
for rank, pi in enumerate(sorted_idx[:5]):
    print(f"  {rank+1}. Policy {pi:3d}: joint_sat={sat_score[pi]:.3f}")

### Interpreting satisficing results

- **Per-objective satisficing rates are each ~50%** by construction -- that is what the median threshold guarantees for individual objectives.
- **Joint satisficing (all objectives simultaneously) is much lower** -- often below 10-20%, even for the best policies. This is a direct consequence of trade-offs.
- **Variation across scenarios matters:** a policy with a satisficing score of 0.4 meets all thresholds in 40% of the 50 climate futures.

---

## Step 4 -- Minimax regret

For each scenario, we compute the best any policy achieves on each objective (the per-scenario ideal), then normalise each policy's performance relative to that ideal. The **maximum regret** across scenarios is the minimax criterion: the worst-case disappointment this policy could produce.

**Task 4.1** -- Implement minimax regret computation in the cell below.

In [ ]:
# Minimax regret: find the worst-case regret across scenarios for each policy.
# Regret[i,j,k] = how much worse policy i is vs. the best policy in scenario j on objective k.
#
# Steps:
#   1. ideal     = np.nanmin(results, axis=0)   # shape (n_scenarios, n_objectives)
#   2. anti_ideal= np.nanmax(results, axis=0)
#   3. rng       = anti_ideal - ideal + 1e-12
#   4. regret    = (results - ideal[np.newaxis,:,:]) / rng[np.newaxis,:,:]   # (P,S,O)
#   5. total_regret = np.nansum(regret, axis=2)   # shape (n_policies, n_scenarios)
#   6. max_regret   = np.nanmax(total_regret, axis=1)  # worst case per policy
#   7. sorted_idx   = np.argsort(max_regret)           # ascending: index 0 = most robust

ideal        = ...
anti_ideal   = ...
rng          = ...
regret       = ...
total_regret = ...
max_regret   = ...
sorted_idx   = ...

print("Top 5 most robust policies (minimax regret):")
for rank, pi in enumerate(sorted_idx[:5]):
    print(f"  {rank+1}. Policy {pi:3d}: max_regret={max_regret[pi]:.3f}  joint_sat={sat_score[pi]:.3f}")

---

## Step 5 -- Visualisation

Three plots:

1. **Satisficing heatmap** -- fraction of scenarios each policy meets each objective threshold.
2. **Regret heatmap** -- maximum normalised regret per objective for all policies sorted by max regret.
3. **CDF of maximum regret** -- the full distribution of worst-case robustness across the Pareto front.

**Task 5.1** -- Complete the two plot cells below.

In [ ]:
# Helper function for imshow-based heatmap (provided — do not modify)
def _draw_heatmap(ax, data, cmap, vmin, vmax, row_labels, col_labels, title, cbar_label):
    """imshow-based heatmap (avoids seaborn/matplotlib 3.14 deepcopy incompatibility)."""
    im = ax.imshow(data, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax,
                   interpolation="nearest")
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, fontsize=9)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=7)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            tc = "white" if v > (vmax - vmin) * 0.6 + vmin else "black"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color=tc)
    plt.colorbar(im, ax=ax, label=cbar_label, shrink=0.8)
    ax.set_title(title, fontsize=10)


# Figure 1 — Satisficing heatmap
# per_obj_sat[i, k] = fraction of scenarios where policy i meets objective k threshold
# Hint: per_obj_sat = np.nanmean(results <= thresholds[np.newaxis, np.newaxis, :], axis=1)

per_obj_sat = ...   # shape (n_policies, n_objectives)
sat_order   = np.argsort(-sat_score)   # descending by joint satisficing

N_SHOW = min(30, N_POLICIES)
fig, ax = plt.subplots(figsize=(6, max(5, N_SHOW * 0.3 + 2)))

# Hint: _draw_heatmap(ax, per_obj_sat[sat_order[:N_SHOW], :], "YlGn", 0, 1, ...)
...

plt.tight_layout()
plt.savefig(os.path.join(_PLOTS_DIR, "a08_satisficing_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: plots/a08_satisficing_heatmap.png")

In [ ]:
# Figure 2 — Regret heatmap + CDF of max regret
N_SHOW = min(30, N_POLICIES)
top_idx_regret = sorted_idx[:N_SHOW]   # sorted by minimax regret (ascending)

worst_scenario_idx   = np.nanargmax(total_regret, axis=1)
worst_regret_per_obj = regret[np.arange(N_POLICIES), worst_scenario_idx, :]

fig, axes = plt.subplots(1, 2, figsize=(14, max(6, N_SHOW * 0.25 + 2)))

# Heatmap: worst-scenario regret per objective for top N_SHOW policies
_draw_heatmap(
    axes[0],
    worst_regret_per_obj[top_idx_regret, :],
    cmap="YlOrRd", vmin=0, vmax=1,
    row_labels=[f"P{pi}" for pi in top_idx_regret],
    col_labels=OBJ_LABELS,
    title="Regret in worst scenario\n(top = most robust by minimax regret)",
    cbar_label="Normalised regret (0=none, 1=worst)",
)

# CDF of max regret
cdf_x = np.sort(max_regret)
cdf_y = np.arange(1, N_POLICIES + 1) / N_POLICIES
axes[1].step(cdf_x, cdf_y, color="steelblue", lw=2)
axes[1].axvline(max_regret[sorted_idx[0]], color="red", lw=1.5, ls="--",
                label=f"Most robust: P{sorted_idx[0]} (regret={max_regret[sorted_idx[0]]:.3f})")
axes[1].set_xlabel("Max total normalised regret (worst scenario)")
axes[1].set_ylabel("CDF")
axes[1].set_title("CDF of minimax regret across policies")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(_PLOTS_DIR, "a08_regret_heatmap.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved.")

### How to read the satisficing heatmap

- **Rows** are policies sorted from highest (top) to lowest (bottom) joint satisficing score.
- **Columns** are the four objectives. Each cell shows the fraction of 50 scenarios in which that policy meets that objective's threshold.
- **Which column is palest?** That objective is the **binding constraint** -- the one most difficult to satisfy simultaneously with the others.

### How to read the regret heatmap and CDF

- **Regret heatmap rows** are the per-policy regret values *in their worst scenario*. Robust policies have pale rows; fragile policies have at least one very dark cell.
- **CDF:** x-axis is worst-case total regret (sum of 4 normalised regrets in the worst scenario). A steep early rise means most policies are similarly robust; a gradual slope indicates genuine spread.

In [ ]:
# Identify the most robust policy (lowest minimax regret)
best_pi  = sorted_idx[0]
best_row = ref_set.iloc[best_pi]

print("Most robust policy (minimax regret):")
print(f"  Policy index          : {best_pi}")
print(f"  Max regret            : {max_regret[best_pi]:.4f}")
print(f"  Satisficing score     : {sat_score[best_pi]:.1%}  "
      f"(satisfices all thresholds in {sat_score[best_pi]*N_SCENARIOS:.0f}/{N_SCENARIOS} scenarios)")
print()

# Mean objective values across 50 scenarios for best_pi
mean_obj = np.nanmean(results[best_pi, :, :], axis=0)
std_obj  = np.nanstd(results[best_pi, :, :], axis=0)

print("Mean objective performance across scenarios:")
for obj, m, s in zip(OBJECTIVES, mean_obj, std_obj):
    print(f"  {obj:35s}: {m:.4g}  ± {s:.4g}")

print("\nFor reference — best single-objective policies:")
for k, obj in enumerate(OBJECTIVES):
    best_val_idx = np.nanargmin(results[:, :, k].mean(axis=1))
    print(f"  Best mean {obj[:15]:15s}: policy {best_val_idx:3d}, mean={results[best_val_idx,:,k].mean():.4g}")

### Interpreting the most robust policy

The minimax-regret policy is **not the best on any single objective** — that would make it an extreme of the Pareto front, not a balanced interior solution. Instead, it achieves:

- **`years_above_2C` below the Pareto-front median** — it achieves meaningful climate protection without being the most aggressive policy.
- **`welfare_loss_abatement` near or below the median** — it does not pay the highest abatement cost.
- **`welfare` near the median** — welfare is nearly invariant across the Pareto front; it discriminates little between policies.
- **A satisficing score that is above zero** — it meets all thresholds simultaneously in at least some futures, unlike the most extreme Pareto solutions.

A risk-averse decision-maker prefers this policy because, *under any of the 50 climate futures, this policy never performs catastrophically badly relative to what was achievable in that future*.

#### Caveats to communicate clearly

1. **Equal weighting assumed.** Total regret sums four objectives with equal weight. A decision-maker who cares much more about climate risk should weight `years_above_2C` more heavily — this would shift the optimal selection toward more climate-aggressive policies.
2. **Scenario coverage is finite.** 50 FAIR members span much more uncertainty than the 10 used during optimisation, but not all 1 000 possible members. A different draw of 50 would give slightly different regret scores.
3. **Pareto front scope.** The reference set was generated by optimising under the Utilitarian welfare function. Policies that are robust under Utilitarian may not be robust under a different welfare function, and the latter's Pareto front was never explored.

---

## Reflection Questions

**1. Why use 50 FAIR ensemble members for re-evaluation rather than the 10 used during optimisation?**


*(Write your answer here.)*


**2. Why is the satisficing score typically low (e.g., 10–30%) even for the best policies?**


*(Write your answer here.)*


**3. Which single policy would you recommend, and why?**


*(Write your answer here.)*